In [3]:
!pip install rapidfuzz openpyxl

Install Libraries

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from rapidfuzz import fuzz
from IPython.display import display

File Paths and settings

In [5]:
INPUT_DIR = Path("outputs/family_evidence")
OUTPUT_DIR = Path("outputs/family_review_evidence")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BAPTISM_FILE = INPUT_DIR / "baptism_family.csv"
MARRIAGE_FILE = INPUT_DIR / "marriage_couples.csv"
BURIAL_FILE = INPUT_DIR / "burial_family.csv"

# Safety settings
MAX_GROUP_SIZE = 250       # avoids huge self-merges from very common names
MAX_EXPORT_ROWS = 100000   # Excel export row limit control

print("Input folder:", INPUT_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

Input folder: C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\family_evidence
Output folder: C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\family_review_evidence


Load family evidence tables

In [6]:
baptism_family = pd.read_csv(BAPTISM_FILE)
marriage_couples = pd.read_csv(MARRIAGE_FILE)
burial_family = pd.read_csv(BURIAL_FILE)

print("BaptismFamily:", baptism_family.shape)
print("MarriageCouples:", marriage_couples.shape)
print("BurialFamily:", burial_family.shape)

display(baptism_family.head())
display(marriage_couples.head())
display(burial_family.head())

BaptismFamily: (6338, 25)
MarriageCouples: (1718, 36)
BurialFamily: (2115, 39)


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,mother_last_name,parent_pair_key,child_parent_key,father_motherfirst_key,mother_fatherfirst_key,has_child,has_father,has_mother,has_parent_pair,has_child_parent_key
0,bautizo-1,1790-10-04,1790.0,pampamarca,domingo ayquipa,lucas ayquipa,sevastiana quispe,persona-1,persona-2,persona-3,...,quispe,lucas ayquipa | sevastiana quispe,domingo ayquipa | lucas ayquipa | sevastiana q...,lucas ayquipa | sevastiana,sevastiana quispe | lucas,True,True,True,True,True
1,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,espinosa,francisco flores | estefa espinosa,andrea flores | francisco flores | estefa espi...,francisco flores | estefa,estefa espinosa | francisco,True,True,True,True,True
2,bautizo-100,1791-11-07,1791.0,unknown,theodoro barrientos,NaN,martina barrientos,persona-376,NaN,persona-377,...,barrientos,NaN,NaN,NaN,NaN,True,False,True,False,False
3,bautizo-1000,1804-03-14,1804.0,aucara,ventura alcari,felipe alcari,ana carrera,persona-3878,persona-3879,persona-3880,...,carrera,felipe alcari | ana carrera,ventura alcari | felipe alcari | ana carrera,felipe alcari | ana,ana carrera | felipe,True,True,True,True,True
4,bautizo-1001,1804-03-20,1804.0,aucara,mariano huaman,damian huaman,eulalia vilcacuri,persona-3882,persona-3883,persona-3884,...,vilcacuri,damian huaman | eulalia vilcacuri,mariano huaman | damian huaman | eulalia vilca...,damian huaman | eulalia,eulalia vilcacuri | damian,True,True,True,True,True


,event_idno,event_date,event_year,event_place_clean,husband_name,husband_father_name,husband_mother_name,wife_name,wife_father_name,wife_mother_name,...,wife_mother_first_name,wife_mother_last_name,spouse_pair_key,husband_wifefirst_key,wife_husbandfirst_key,husband_parent_pair_key,wife_parent_pair_key,has_husband,has_wife,has_spouse_pair
0,matrimonio-1,1816-12-06,1816.0,aucara,jose manl manuel de la roca,acencio roca,leonor guerrero,juana rodrigues,pedro rodrigues,magdalena sotelo,...,magdalena,sotelo,jose manl manuel de la roca | juana rodrigues,jose manl manuel de la roca | juana,juana rodrigues | jose,acencio roca | leonor guerrero,pedro rodrigues | magdalena sotelo,True,True,True
1,matrimonio-10,1817-06-20,1817.0,ishua santa iglesia,patricio asto,mateo asto,bernarda ramos,melchora sebidola,cipriano sebedola,grabiela guamani,...,grabiela,guamani,patricio asto | melchora sebidola,patricio asto | melchora,melchora sebidola | patricio,mateo asto | bernarda ramos,cipriano sebedola | grabiela guamani,True,True,True
2,matrimonio-100,1820-08-02,1820.0,santa iglesia,pedro huallpatuiru,NaN,eulalia sanches,eugenia barrientos,fernando barrientos,nolberta puma,...,nolberta,puma,pedro huallpatuiru | eugenia barrientos,pedro huallpatuiru | eugenia,eugenia barrientos | pedro,NaN,fernando barrientos | nolberta puma,True,True,True
3,matrimonio-1000,1862-04-04,1862.0,aucara,julian sanches,jose sanches,leocadia llacsamanta,josefa ramos,tiburcio ramos,rosa quispe,...,rosa,quispe,julian sanches | josefa ramos,julian sanches | josefa,josefa ramos | julian,jose sanches | leocadia llacsamanta,tiburcio ramos | rosa quispe,True,True,True
4,matrimonio-1001,1862-04-04,1862.0,aucara,mariano flores,eucebio flores,maria ynca,maria espinosa,manuel espinosa,hermenejilda luca,...,hermenejilda,luca,mariano flores | maria espinosa,mariano flores | maria,maria espinosa | mariano,eucebio flores | maria ynca,manuel espinosa | hermenejilda luca,True,True,True


,event_idno,event_date,event_year,event_place_clean,deceased_name,father_name,husband_name,mother_name,wife_name,deceased_persona_id,...,father_motherfirst_key,mother_fatherfirst_key,has_deceased,has_father,has_mother,has_husband,has_wife,has_spouse,has_deceased_parent_key,has_deceased_spouse_key
0,entierro-1,1846-10-06,1846.0,unknown,julian xavies,NaN,NaN,NaN,mercedes lupa,persona-41678,...,NaN,NaN,True,False,False,False,True,True,False,True
1,entierro-10,1848-11-17,1848.0,unknown,manuela hermosa,juan hermosa,NaN,clemencia manus,NaN,persona-41695,...,juan hermosa | clemencia,clemencia manus | juan,True,True,True,False,False,False,True,False
2,entierro-100,1866-05-04,1866.0,unknown,juan sanchez,juan sanchez,NaN,cipriana quispe,NaN,persona-41909,...,juan sanchez | cipriana,cipriana quispe | juan,True,True,True,False,False,False,True,False
3,entierro-1000,1895-12-30,1895.0,unknown,jacova misaguel unknown,NaN,NaN,NaN,NaN,persona-43648,...,NaN,NaN,True,False,False,False,False,False,False,False
4,entierro-1001,1895-11-27,1895.0,unknown,melchor rojas,NaN,NaN,NaN,NaN,persona-43649,...,NaN,NaN,True,False,False,False,False,False,False,False


Helper functions for texts and names


In [7]:
def is_missing(value):
    return pd.isna(value) or str(value).strip() == "" or str(value).strip().lower() in ["nan", "none"]


def safe_str(value):
    if is_missing(value):
        return np.nan
    return str(value).strip().lower()


def first_token(value):
    if is_missing(value):
        return np.nan
    parts = str(value).strip().split()
    return parts[0] if len(parts) > 0 else np.nan


def last_token(value):
    if is_missing(value):
        return np.nan
    parts = str(value).strip().split()
    return parts[-1] if len(parts) > 1 else np.nan


def make_key(*values):
    clean_values = []
    for value in values:
        if is_missing(value):
            return np.nan
        clean_values.append(str(value).strip())
    return " | ".join(clean_values)


def ensure_column(df, col):
    if col not in df.columns:
        df[col] = np.nan
    return df


def ensure_columns(df, cols):
    for col in cols:
        ensure_column(df, col)
    return df

Ensure first/last name columns exist

In [8]:
def ensure_name_parts(df, roles):
    """
    Creates first-name and last-name columns for each role if they do not already exist.
    Example:
    father_name → father_first_name, father_last_name
    """
    df = df.copy()
    
    for role in roles:
        name_col = f"{role}_name"
        first_col = f"{role}_first_name"
        last_col = f"{role}_last_name"
        
        if name_col in df.columns:
            if first_col not in df.columns:
                df[first_col] = df[name_col].apply(first_token)
            if last_col not in df.columns:
                df[last_col] = df[name_col].apply(last_token)
    
    return df


baptism_family = ensure_name_parts(baptism_family, ["child", "father", "mother"])
marriage_couples = ensure_name_parts(
    marriage_couples,
    ["husband", "wife", "husband_father", "husband_mother", "wife_father", "wife_mother"]
)
burial_family = ensure_name_parts(burial_family, ["deceased", "father", "mother", "husband", "wife", "spouse"])

print("Name-part columns checked.")

Name-part columns checked.


Fuzzy similarity and name relationship logic

In [9]:
def similarity_score(a, b):
    """
    Returns fuzzy similarity score between 0 and 1.
    """
    if is_missing(a) or is_missing(b):
        return np.nan
    
    return fuzz.WRatio(str(a), str(b)) / 100


def name_relation(a, b, close_threshold=0.88):
    """
    Classifies relationship between two names.

    Categories:
    - exact_same
    - close_variant
    - same_first_name
    - same_last_name
    - clearly_different
    - unknown
    """
    if is_missing(a) or is_missing(b):
        return "unknown"
    
    a = str(a).strip().lower()
    b = str(b).strip().lower()
    
    if a == b:
        return "exact_same"
    
    overall = similarity_score(a, b)
    
    a_first = first_token(a)
    b_first = first_token(b)
    a_last = last_token(a)
    b_last = last_token(b)
    
    first_sim = similarity_score(a_first, b_first)
    last_sim = similarity_score(a_last, b_last)
    
    # Strong overall spelling similarity
    if pd.notna(overall) and overall >= close_threshold:
        return "close_variant"
    
    # First and last are both close
    if pd.notna(first_sim) and pd.notna(last_sim):
        if first_sim >= 0.88 and last_sim >= 0.80:
            return "close_variant"
    
    # Same or very close first name
    if not is_missing(a_first) and not is_missing(b_first):
        if a_first == b_first:
            return "same_first_name"
        if pd.notna(first_sim) and first_sim >= 0.92:
            return "same_first_name"
    
    # Same or very close last name
    if not is_missing(a_last) and not is_missing(b_last):
        if a_last == b_last:
            return "same_last_name"
        if pd.notna(last_sim) and last_sim >= 0.92:
            return "same_last_name"
    
    return "clearly_different"


def add_name_comparison(df, left_col, right_col, prefix):
    """
    Adds fuzzy similarity and relationship category between two name columns.
    """
    df = df.copy()
    
    if left_col not in df.columns:
        df[left_col] = np.nan
    if right_col not in df.columns:
        df[right_col] = np.nan
    
    df[f"{prefix}_similarity"] = [
        similarity_score(a, b) for a, b in zip(df[left_col], df[right_col])
    ]
    
    df[f"{prefix}_relation"] = [
        name_relation(a, b) for a, b in zip(df[left_col], df[right_col])
    ]
    
    return df

Year gap logic

In [10]:
def add_year_features(df, left_year_col="event_year", right_year_col="event_year_r"):
    """
    Adds absolute and signed year differences.
    """
    df = df.copy()
    
    if left_year_col not in df.columns:
        df[left_year_col] = np.nan
    if right_year_col not in df.columns:
        df[right_year_col] = np.nan
    
    left_year = pd.to_numeric(df[left_year_col], errors="coerce")
    right_year = pd.to_numeric(df[right_year_col], errors="coerce")
    
    df["year_gap"] = (left_year - right_year).abs()
    df["signed_year_diff"] = right_year - left_year
    
    return df


def child_year_gap_category(gap):
    """
    Professor feedback:
    0–2 years = likely same child
    10+ years = likely sibling name reuse or homonym
    """
    if pd.isna(gap):
        return "unknown"
    elif gap <= 2:
        return "strong_child_match_window"
    elif gap <= 5:
        return "possible_child_match_window"
    elif gap <= 9:
        return "weak_child_match_window"
    else:
        return "likely_sibling_or_homonym"


def parent_year_gap_category(gap):
    """
    Professor feedback:
    same parent couple within about 20 years is very strong.
    """
    if pd.isna(gap):
        return "unknown"
    elif gap <= 20:
        return "strong_parent_window"
    elif gap <= 30:
        return "parent_review_window"
    else:
        return "weak_parent_window"


def marriage_year_gap_category(gap):
    if pd.isna(gap):
        return "unknown"
    elif gap <= 5:
        return "possible_duplicate_marriage"
    elif gap <= 20:
        return "review_window"
    else:
        return "long_gap_review"

Self-merge helper for review pairs

In [11]:
large_group_reports = {}


def make_pair_key(df, left_event_col="event_idno", right_event_col="event_idno_r"):
    """
    Creates a stable pair key so A-B and B-A are treated as the same pair.
    """
    left = df[left_event_col].astype(str)
    right = df[right_event_col].astype(str)
    
    return np.where(
        left < right,
        left + " | " + right,
        right + " | " + left
    )


def self_merge_pairs(df, key_col, review_name, max_group_size=MAX_GROUP_SIZE):
    """
    Creates candidate pairs by self-merging a table on a blocking key.
    Removes:
    - null keys
    - groups that are too large
    - same-event pairs
    - mirror duplicates
    """
    if key_col not in df.columns:
        print(f"{review_name}: key column missing → {key_col}")
        return pd.DataFrame()
    
    temp = df.copy()
    temp = temp[temp[key_col].notna()].copy()
    
    if temp.empty:
        print(f"{review_name}: no non-null keys for {key_col}")
        return pd.DataFrame()
    
    group_sizes = temp[key_col].value_counts().reset_index()
    group_sizes.columns = [key_col, "group_size"]
    
    large_groups = group_sizes[group_sizes["group_size"] > max_group_size].copy()
    allowed_keys = group_sizes[
        (group_sizes["group_size"] >= 2) &
        (group_sizes["group_size"] <= max_group_size)
    ][key_col]
    
    large_group_reports[review_name] = large_groups
    
    temp = temp[temp[key_col].isin(allowed_keys)].copy()
    
    if temp.empty:
        print(f"{review_name}: no usable groups after group-size filtering")
        return pd.DataFrame()
    
    pairs = temp.merge(
        temp,
        on=key_col,
        how="inner",
        suffixes=("", "_r")
    )
    
    # Remove same event
    if "event_idno" in pairs.columns and "event_idno_r" in pairs.columns:
        pairs = pairs[pairs["event_idno"] != pairs["event_idno_r"]].copy()
    
    if pairs.empty:
        print(f"{review_name}: no different-event pairs")
        return pd.DataFrame()
    
    # Remove mirror duplicates
    pairs["pair_key"] = make_pair_key(pairs)
    pairs = pairs.drop_duplicates(subset=["pair_key"]).copy()
    
    print(f"{review_name}: {pairs.shape[0]} review pairs created using {key_col}")
    
    return pairs

Baptism Review 1: Child duplicate candidates

In [12]:
child_duplicates = self_merge_pairs(
    baptism_family,
    key_col="child_parent_key",
    review_name="Child_Duplicate_Candidates",
    max_group_size=MAX_GROUP_SIZE
)

if not child_duplicates.empty:
    child_duplicates = add_year_features(child_duplicates)
    
    child_duplicates = add_name_comparison(
        child_duplicates, "child_name", "child_name_r", "child"
    )
    child_duplicates = add_name_comparison(
        child_duplicates, "father_name", "father_name_r", "father"
    )
    child_duplicates = add_name_comparison(
        child_duplicates, "mother_name", "mother_name_r", "mother"
    )
    
    child_duplicates["year_gap_category"] = child_duplicates["year_gap"].apply(child_year_gap_category)
    
    child_duplicates["child_duplicate_decision"] = np.select(
        [
            child_duplicates["year_gap"].le(2),
            child_duplicates["year_gap"].between(3, 5, inclusive="both"),
            child_duplicates["year_gap"].between(6, 9, inclusive="both"),
            child_duplicates["year_gap"].ge(10)
        ],
        [
            "high_confidence_same_child",
            "possible_same_child_review",
            "weak_same_child_review",
            "likely_sibling_name_reuse_or_homonym"
        ],
        default="unknown"
    )

display(child_duplicates.head())

Child_Duplicate_Candidates: 21 review pairs created using child_parent_key


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,year_gap,signed_year_diff,child_similarity,child_relation,father_similarity,father_relation,mother_similarity,mother_relation,year_gap_category,child_duplicate_decision
1,bautizo-1342,1809-12-07,1809.0,aucara,maria sanchez,agustin sanchez,maria barrientos,persona-5220,persona-5221,persona-5222,...,11.0,-11.0,1.0,exact_same,1.0,exact_same,1.0,exact_same,likely_sibling_or_homonym,likely_sibling_name_reuse_or_homonym
3,bautizo-1374,1810-02-15,1810.0,aucara,benito condori,lorenso condori,maria cruz,persona-5344,persona-5345,persona-5346,...,0.0,0.0,1.0,exact_same,1.0,exact_same,1.0,exact_same,strong_child_match_window,high_confidence_same_child
7,bautizo-1774,1815-02-23,1815.0,pampamarca,petrona cruz,francisco cruz,victoria quispe,persona-6920,persona-6921,persona-6922,...,22.0,-22.0,1.0,exact_same,1.0,exact_same,1.0,exact_same,likely_sibling_or_homonym,likely_sibling_name_reuse_or_homonym
9,bautizo-2225,1828-12-19,1828.0,pampamarca santa iglesia,jose condori condori,raymundo condori,manuela tito,persona-8679,persona-8680,persona-8681,...,1.0,1.0,1.0,exact_same,1.0,exact_same,1.0,exact_same,strong_child_match_window,high_confidence_same_child
11,bautizo-2228,1828-12-19,1828.0,pampamarca santa iglesia,josefa sanches sanches,leonardo sanches,evarista ccoyllo,persona-8691,persona-8692,persona-8693,...,1.0,1.0,1.0,exact_same,1.0,exact_same,1.0,exact_same,strong_child_match_window,high_confidence_same_child


Baptism Review 2: SameParentCouple

In [13]:
same_parent_couple = self_merge_pairs(
    baptism_family,
    key_col="parent_pair_key",
    review_name="SameParentCouple",
    max_group_size=MAX_GROUP_SIZE
)

if not same_parent_couple.empty:
    same_parent_couple = add_year_features(same_parent_couple)
    
    same_parent_couple = add_name_comparison(
        same_parent_couple, "child_name", "child_name_r", "child"
    )
    
    same_parent_couple["parent_year_gap_category"] = same_parent_couple["year_gap"].apply(parent_year_gap_category)
    
    same_parent_couple["parent_match_decision"] = np.select(
        [
            same_parent_couple["year_gap"].le(20),
            same_parent_couple["year_gap"].between(21, 30, inclusive="both"),
            same_parent_couple["year_gap"].gt(30)
        ],
        [
            "very_strong_same_parent_couple",
            "same_parent_couple_review_window",
            "weak_due_to_long_parent_window"
        ],
        default="unknown"
    )
    
    # Keep professor-relevant parent window
    same_parent_couple = same_parent_couple[same_parent_couple["year_gap"] <= 30].copy()

display(same_parent_couple.head())

SameParentCouple: 1263 review pairs created using parent_pair_key


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,has_mother_r,has_parent_pair_r,has_child_parent_key_r,pair_key,year_gap,signed_year_diff,child_similarity,child_relation,parent_year_gap_category,parent_match_decision
1,bautizo-1003,1804-03-30,1804.0,pampamarca,enrrique huauya,pedro huauya,tomasa barrientos,persona-3890,persona-3891,persona-3892,...,True,True,True,bautizo-1003 | bautizo-1043,1.0,1.0,0.666667,same_last_name,strong_parent_window,very_strong_same_parent_couple
2,bautizo-1003,1804-03-30,1804.0,pampamarca,enrrique huauya,pedro huauya,tomasa barrientos,persona-3890,persona-3891,persona-3892,...,True,True,True,bautizo-1003 | bautizo-1295,5.0,5.0,0.666667,same_last_name,strong_parent_window,very_strong_same_parent_couple
4,bautizo-1006,1804-04-20,1804.0,pampamarca,maria quispe,manuel quispe,estefa hualpatucro,persona-3902,persona-3903,persona-3904,...,True,True,True,bautizo-1006 | bautizo-549,8.0,-8.0,0.692308,same_last_name,strong_parent_window,very_strong_same_parent_couple
5,bautizo-1006,1804-04-20,1804.0,pampamarca,maria quispe,manuel quispe,estefa hualpatucro,persona-3902,persona-3903,persona-3904,...,True,True,True,bautizo-1006 | bautizo-793,5.0,-5.0,0.740741,same_last_name,strong_parent_window,very_strong_same_parent_couple
7,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,bautizo-1008 | bautizo-1412,6.0,6.0,0.689655,same_last_name,strong_parent_window,very_strong_same_parent_couple


Baptism Review 3: Father_SimilarMother

In [14]:
father_similar_mother = self_merge_pairs(
    baptism_family,
    key_col="father_motherfirst_key",
    review_name="Father_SimilarMother",
    max_group_size=MAX_GROUP_SIZE
)

if not father_similar_mother.empty:
    father_similar_mother = add_year_features(father_similar_mother)
    
    # Keep same father, but mother full name differs
    father_similar_mother = father_similar_mother[
        (father_similar_mother["father_name"] == father_similar_mother["father_name_r"]) &
        (father_similar_mother["mother_name"] != father_similar_mother["mother_name_r"]) &
        (father_similar_mother["year_gap"] <= 30)
    ].copy()
    
    father_similar_mother = add_name_comparison(
        father_similar_mother, "mother_name", "mother_name_r", "mother"
    )
    
    father_similar_mother["review_decision"] = np.where(
        father_similar_mother["mother_relation"].isin(
            ["close_variant", "same_first_name", "same_last_name"]
        ),
        "likely_same_mother_name_variant",
        "review_needed_possible_different_mother"
    )

display(father_similar_mother.head())

Father_SimilarMother: 2320 review pairs created using father_motherfirst_key


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,has_father_r,has_mother_r,has_parent_pair_r,has_child_parent_key_r,pair_key,year_gap,signed_year_diff,mother_similarity,mother_relation,review_decision
1,bautizo-1000,1804-03-14,1804.0,aucara,ventura alcari,felipe alcari,ana carrera,persona-3878,persona-3879,persona-3880,...,True,True,True,True,bautizo-1000 | bautizo-1420,6.0,6.0,0.636364,same_first_name,likely_same_mother_name_variant
3,bautizo-1002,1804-03-25,1804.0,pampamarca,marcelo espilco,fernando espilco,rosa guarcaya,persona-3886,persona-3887,persona-3888,...,True,True,True,True,bautizo-1002 | bautizo-1327,5.0,5.0,0.640000,same_first_name,likely_same_mother_name_variant
4,bautizo-1002,1804-03-25,1804.0,pampamarca,marcelo espilco,fernando espilco,rosa guarcaya,persona-3886,persona-3887,persona-3888,...,True,True,True,True,bautizo-1002 | bautizo-1343,5.0,5.0,0.888889,close_variant,likely_same_mother_name_variant
5,bautizo-1002,1804-03-25,1804.0,pampamarca,marcelo espilco,fernando espilco,rosa guarcaya,persona-3886,persona-3887,persona-3888,...,True,True,True,True,bautizo-1002 | bautizo-676,7.0,-7.0,0.923077,close_variant,likely_same_mother_name_variant
6,bautizo-1002,1804-03-25,1804.0,pampamarca,marcelo espilco,fernando espilco,rosa guarcaya,persona-3886,persona-3887,persona-3888,...,True,True,True,True,bautizo-1002 | bautizo-811,5.0,-5.0,0.846154,same_first_name,likely_same_mother_name_variant


Baptism Review 4: Mother_SimilarFather

In [15]:
mother_similar_father = self_merge_pairs(
    baptism_family,
    key_col="mother_fatherfirst_key",
    review_name="Mother_SimilarFather",
    max_group_size=MAX_GROUP_SIZE
)

if not mother_similar_father.empty:
    mother_similar_father = add_year_features(mother_similar_father)
    
    # Keep same mother, but father full name differs
    mother_similar_father = mother_similar_father[
        (mother_similar_father["mother_name"] == mother_similar_father["mother_name_r"]) &
        (mother_similar_father["father_name"] != mother_similar_father["father_name_r"]) &
        (mother_similar_father["year_gap"] <= 30)
    ].copy()
    
    mother_similar_father = add_name_comparison(
        mother_similar_father, "father_name", "father_name_r", "father"
    )
    
    mother_similar_father["review_decision"] = np.where(
        mother_similar_father["father_relation"].isin(
            ["close_variant", "same_first_name", "same_last_name"]
        ),
        "likely_same_father_name_variant",
        "review_needed_possible_different_father"
    )

display(mother_similar_father.head())

Mother_SimilarFather: 2024 review pairs created using mother_fatherfirst_key


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,has_father_r,has_mother_r,has_parent_pair_r,has_child_parent_key_r,pair_key,year_gap,signed_year_diff,father_similarity,father_relation,review_decision
9,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-370,10.0,-10.0,0.933333,close_variant,likely_same_father_name_variant
11,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-957,1.0,-1.0,0.933333,close_variant,likely_same_father_name_variant
13,bautizo-101,1791-11-07,1791.0,unknown,carlos huauia,francisco huauia,casimira coyllo,persona-379,persona-380,persona-381,...,True,True,True,True,bautizo-101 | bautizo-1098,16.0,16.0,0.937500,close_variant,likely_same_father_name_variant
14,bautizo-101,1791-11-07,1791.0,unknown,carlos huauia,francisco huauia,casimira coyllo,persona-379,persona-380,persona-381,...,True,True,True,True,bautizo-101 | bautizo-1480,20.0,20.0,0.937500,close_variant,likely_same_father_name_variant
15,bautizo-101,1791-11-07,1791.0,unknown,carlos huauia,francisco huauia,casimira coyllo,persona-379,persona-380,persona-381,...,True,True,True,True,bautizo-101 | bautizo-641,6.0,6.0,0.937500,close_variant,likely_same_father_name_variant


Baptism Review 5: Father_DifferentMother

In [16]:
father_different_mother = self_merge_pairs(
    baptism_family,
    key_col="father_name",
    review_name="Father_DifferentMother",
    max_group_size=150
)

if not father_different_mother.empty:
    father_different_mother = add_year_features(father_different_mother)
    
    father_different_mother = father_different_mother[
        father_different_mother["mother_name"].notna() &
        father_different_mother["mother_name_r"].notna() &
        (father_different_mother["mother_name"] != father_different_mother["mother_name_r"]) &
        (father_different_mother["year_gap"] <= 40)
    ].copy()
    
    father_different_mother = add_name_comparison(
        father_different_mother, "mother_name", "mother_name_r", "mother"
    )
    
    father_different_mother["review_decision"] = np.where(
        father_different_mother["mother_relation"] == "clearly_different",
        "true_different_mother_or_partner_candidate",
        "likely_same_mother_variant_not_true_different"
    )

display(father_different_mother.head())

Father_DifferentMother: 8752 review pairs created using father_name


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,has_father_r,has_mother_r,has_parent_pair_r,has_child_parent_key_r,pair_key,year_gap,signed_year_diff,mother_similarity,mother_relation,review_decision
1,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,True,True,True,True,bautizo-10 | bautizo-356,4.0,4.0,0.322581,clearly_different,true_different_mother_or_partner_candidate
2,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,True,True,True,True,bautizo-10 | bautizo-589,6.0,6.0,0.327586,clearly_different,true_different_mother_or_partner_candidate
3,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,True,True,True,True,bautizo-10 | bautizo-732,8.0,8.0,0.327586,clearly_different,true_different_mother_or_partner_candidate
4,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,True,True,True,True,bautizo-10 | bautizo-870,11.0,11.0,0.285714,clearly_different,true_different_mother_or_partner_candidate
5,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,True,True,True,True,bautizo-10 | bautizo-970,13.0,13.0,0.327586,clearly_different,true_different_mother_or_partner_candidate


Baptism Review 6: Mother_DifferentFather

In [17]:
mother_different_father = self_merge_pairs(
    baptism_family,
    key_col="mother_name",
    review_name="Mother_DifferentFather",
    max_group_size=150
)

if not mother_different_father.empty:
    mother_different_father = add_year_features(mother_different_father)
    
    mother_different_father = mother_different_father[
        mother_different_father["father_name"].notna() &
        mother_different_father["father_name_r"].notna() &
        (mother_different_father["father_name"] != mother_different_father["father_name_r"]) &
        (mother_different_father["year_gap"] <= 35)
    ].copy()
    
    mother_different_father = add_name_comparison(
        mother_different_father, "father_name", "father_name_r", "father"
    )
    
    mother_different_father["review_decision"] = np.where(
        mother_different_father["father_relation"] == "clearly_different",
        "true_different_father_or_partner_candidate",
        "likely_same_father_variant_not_true_different"
    )

display(mother_different_father.head())

Mother_DifferentFather: 6931 review pairs created using mother_name


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,has_father_r,has_mother_r,has_parent_pair_r,has_child_parent_key_r,pair_key,year_gap,signed_year_diff,father_similarity,father_relation,review_decision
5,bautizo-1002,1804-03-25,1804.0,pampamarca,marcelo espilco,fernando espilco,rosa guarcaya,persona-3886,persona-3887,persona-3888,...,True,True,True,True,bautizo-1002 | bautizo-840,4.0,-4.0,0.727273,clearly_different,true_different_father_or_partner_candidate
14,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-1385,6.0,6.0,0.933333,close_variant,likely_same_father_variant_not_true_different
17,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-370,10.0,-10.0,0.933333,close_variant,likely_same_father_variant_not_true_different
19,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-849,4.0,-4.0,0.933333,close_variant,likely_same_father_variant_not_true_different
20,bautizo-1008,1804-04-28,1804.0,pampamarca,nolverta condori,eugenio condori,andrea tito,persona-3910,persona-3911,persona-3912,...,True,True,True,True,bautizo-1008 | bautizo-957,1.0,-1.0,0.933333,close_variant,likely_same_father_variant_not_true_different


Marriage review evidence

In [18]:
# Same husband + same wife
same_marriage_couple = self_merge_pairs(
    marriage_couples,
    key_col="spouse_pair_key",
    review_name="SameMarriageCouple",
    max_group_size=MAX_GROUP_SIZE
)

if not same_marriage_couple.empty:
    same_marriage_couple = add_year_features(same_marriage_couple)
    same_marriage_couple = same_marriage_couple[same_marriage_couple["year_gap"] <= 5].copy()
    same_marriage_couple["marriage_decision"] = "possible_duplicate_or_repeated_marriage_record"


# Same husband + similar wife
husband_similar_wife = self_merge_pairs(
    marriage_couples,
    key_col="husband_wifefirst_key",
    review_name="Husband_SimilarWife",
    max_group_size=MAX_GROUP_SIZE
)

if not husband_similar_wife.empty:
    husband_similar_wife = add_year_features(husband_similar_wife)
    husband_similar_wife = husband_similar_wife[
        (husband_similar_wife["husband_name"] == husband_similar_wife["husband_name_r"]) &
        (husband_similar_wife["wife_name"] != husband_similar_wife["wife_name_r"]) &
        (husband_similar_wife["year_gap"] <= 20)
    ].copy()
    husband_similar_wife = add_name_comparison(
        husband_similar_wife, "wife_name", "wife_name_r", "wife"
    )
    husband_similar_wife["review_decision"] = np.where(
        husband_similar_wife["wife_relation"].isin(["close_variant", "same_first_name", "same_last_name"]),
        "likely_same_wife_name_variant",
        "review_needed_possible_different_wife"
    )


# Same wife + similar husband
wife_similar_husband = self_merge_pairs(
    marriage_couples,
    key_col="wife_husbandfirst_key",
    review_name="Wife_SimilarHusband",
    max_group_size=MAX_GROUP_SIZE
)

if not wife_similar_husband.empty:
    wife_similar_husband = add_year_features(wife_similar_husband)
    wife_similar_husband = wife_similar_husband[
        (wife_similar_husband["wife_name"] == wife_similar_husband["wife_name_r"]) &
        (wife_similar_husband["husband_name"] != wife_similar_husband["husband_name_r"]) &
        (wife_similar_husband["year_gap"] <= 20)
    ].copy()
    wife_similar_husband = add_name_comparison(
        wife_similar_husband, "husband_name", "husband_name_r", "husband"
    )
    wife_similar_husband["review_decision"] = np.where(
        wife_similar_husband["husband_relation"].isin(["close_variant", "same_first_name", "same_last_name"]),
        "likely_same_husband_name_variant",
        "review_needed_possible_different_husband"
    )


# Same husband + different wife
husband_different_wife = self_merge_pairs(
    marriage_couples,
    key_col="husband_name",
    review_name="Husband_DifferentWife",
    max_group_size=150
)

if not husband_different_wife.empty:
    husband_different_wife = add_year_features(husband_different_wife)
    husband_different_wife = husband_different_wife[
        husband_different_wife["wife_name"].notna() &
        husband_different_wife["wife_name_r"].notna() &
        (husband_different_wife["wife_name"] != husband_different_wife["wife_name_r"]) &
        (husband_different_wife["year_gap"] <= 40)
    ].copy()
    husband_different_wife = add_name_comparison(
        husband_different_wife, "wife_name", "wife_name_r", "wife"
    )
    husband_different_wife["review_decision"] = np.where(
        husband_different_wife["wife_relation"] == "clearly_different",
        "true_different_wife_or_remarriage_candidate",
        "likely_same_wife_variant_not_true_different"
    )


# Same wife + different husband
wife_different_husband = self_merge_pairs(
    marriage_couples,
    key_col="wife_name",
    review_name="Wife_DifferentHusband",
    max_group_size=150
)

if not wife_different_husband.empty:
    wife_different_husband = add_year_features(wife_different_husband)
    wife_different_husband = wife_different_husband[
        wife_different_husband["husband_name"].notna() &
        wife_different_husband["husband_name_r"].notna() &
        (wife_different_husband["husband_name"] != wife_different_husband["husband_name_r"]) &
        (wife_different_husband["year_gap"] <= 35)
    ].copy()
    wife_different_husband = add_name_comparison(
        wife_different_husband, "husband_name", "husband_name_r", "husband"
    )
    wife_different_husband["review_decision"] = np.where(
        wife_different_husband["husband_relation"] == "clearly_different",
        "true_different_husband_or_remarriage_candidate",
        "likely_same_husband_variant_not_true_different"
    )

print("Marriage review evidence created.")

SameMarriageCouple: 12 review pairs created using spouse_pair_key
Husband_SimilarWife: 21 review pairs created using husband_wifefirst_key
Wife_SimilarHusband: 15 review pairs created using wife_husbandfirst_key
Husband_DifferentWife: 327 review pairs created using husband_name
Wife_DifferentHusband: 254 review pairs created using wife_name
Marriage review evidence created.


Baptism ↔ Marriage cross-evidence helper functions

In [22]:
def age_at_marriage_category(age):
    """
    Used when comparing:
    baptized child -> later husband/wife in marriage.
    
    Since baptism year is close to birth year, marriage_year - baptism_year
    is approximately age at marriage.
    """
    if pd.isna(age):
        return "unknown"
    elif age < 10:
        return "unlikely_too_young"
    elif age <= 17:
        return "possible_early_marriage"
    elif age <= 45:
        return "strong_plausible_marriage_age"
    elif age <= 70:
        return "possible_late_marriage"
    else:
        return "unlikely_too_old"


def marriage_to_baptism_gap_category(gap):
    """
    Used when comparing:
    marriage couple -> later baptism parents.
    
    baptism_year - marriage_year tells whether children appear after marriage.
    """
    if pd.isna(gap):
        return "unknown"
    elif gap < -5:
        return "unlikely_baptism_long_before_marriage"
    elif gap < 0:
        return "possible_child_before_marriage_review"
    elif gap <= 25:
        return "strong_childbearing_window"
    elif gap <= 40:
        return "possible_late_childbearing_window"
    else:
        return "weak_due_to_long_gap"


def count_parent_support(row, father_sim_col, mother_sim_col, threshold=0.80):
    """
    Counts how many parent comparisons support the match.
    """
    support = 0
    
    father_sim = row.get(father_sim_col)
    mother_sim = row.get(mother_sim_col)
    
    if pd.notna(father_sim) and father_sim >= threshold:
        support += 1
        
    if pd.notna(mother_sim) and mother_sim >= threshold:
        support += 1
        
    return support

Baptized child later appears as husband or wife

In [23]:
def build_baptism_to_marriage_spouse_candidates(
    baptism_df,
    marriage_df,
    spouse_role
):
    """
    Finds candidates where a baptized child later appears as:
    - husband in marriage
    - wife in marriage
    
    spouse_role must be either 'husband' or 'wife'.
    """
    
    assert spouse_role in ["husband", "wife"], "spouse_role must be 'husband' or 'wife'"
    
    b = baptism_df.copy()
    m = marriage_df.copy()
    
    spouse_name_col = f"{spouse_role}_name"
    spouse_first_col = f"{spouse_role}_first_name"
    spouse_last_col = f"{spouse_role}_last_name"
    spouse_father_col = f"{spouse_role}_father_name"
    spouse_mother_col = f"{spouse_role}_mother_name"
    spouse_father_first_col = f"{spouse_role}_father_first_name"
    spouse_mother_first_col = f"{spouse_role}_mother_first_name"
    
    # Make sure optional marriage-parent columns exist
    for col in [
        spouse_name_col,
        spouse_first_col,
        spouse_last_col,
        spouse_father_col,
        spouse_mother_col,
        spouse_father_first_col,
        spouse_mother_first_col
    ]:
        if col not in m.columns:
            m[col] = np.nan
    
    # Strategy 1: Strong family-context blocking
    # child first + father first + mother first
    b["child_parent_first_key"] = b.apply(
        lambda x: make_key(
            x.get("child_first_name"),
            x.get("father_first_name"),
            x.get("mother_first_name")
        ),
        axis=1
    )
    
    m[f"{spouse_role}_parent_first_key"] = m.apply(
        lambda x: make_key(
            x.get(spouse_first_col),
            x.get(spouse_father_first_col),
            x.get(spouse_mother_first_col)
        ),
        axis=1
    )
    
    b_family = b[b["child_parent_first_key"].notna()].copy()
    m_family = m[m[f"{spouse_role}_parent_first_key"].notna()].copy()
    
    family_candidates = b_family.merge(
        m_family,
        left_on="child_parent_first_key",
        right_on=f"{spouse_role}_parent_first_key",
        how="inner",
        suffixes=("_baptism", "_marriage")
    )
    
    if not family_candidates.empty:
        family_candidates["blocking_strategy"] = "child_parent_first_key"
    
    # Strategy 2: Broader name-only blocking
    # child first + child last
    b["child_name_key"] = b.apply(
        lambda x: make_key(x.get("child_first_name"), x.get("child_last_name")),
        axis=1
    )
    
    m[f"{spouse_role}_name_key"] = m.apply(
        lambda x: make_key(x.get(spouse_first_col), x.get(spouse_last_col)),
        axis=1
    )
    
    b_name = b[b["child_name_key"].notna()].copy()
    m_name = m[m[f"{spouse_role}_name_key"].notna()].copy()
    
    name_candidates = b_name.merge(
        m_name,
        left_on="child_name_key",
        right_on=f"{spouse_role}_name_key",
        how="inner",
        suffixes=("_baptism", "_marriage")
    )
    
    if not name_candidates.empty:
        name_candidates["blocking_strategy"] = "child_name_key"
    
    # Combine both candidate sets
    candidates = pd.concat(
        [family_candidates, name_candidates],
        ignore_index=True
    )
    
    if candidates.empty:
        return candidates
    
    # Remove duplicate baptism-marriage pairs
    candidates["cross_pair_key"] = (
        candidates["event_idno_baptism"].astype(str)
        + " | "
        + candidates["event_idno_marriage"].astype(str)
        + f" | child_to_{spouse_role}"
    )
    
    candidates = candidates.drop_duplicates(subset=["cross_pair_key"]).copy()
    
    # Year logic
    candidates["age_at_marriage"] = (
        pd.to_numeric(candidates["event_year_marriage"], errors="coerce")
        - pd.to_numeric(candidates["event_year_baptism"], errors="coerce")
    )
    
    candidates["age_at_marriage_category"] = candidates["age_at_marriage"].apply(
        age_at_marriage_category
    )
    
    # Name comparisons
    candidates = add_name_comparison(
        candidates,
        "child_name",
        spouse_name_col,
        f"child_{spouse_role}"
    )
    
    candidates = add_name_comparison(
        candidates,
        "father_name",
        spouse_father_col,
        f"baptism_father_{spouse_role}_father"
    )
    
    candidates = add_name_comparison(
        candidates,
        "mother_name",
        spouse_mother_col,
        f"baptism_mother_{spouse_role}_mother"
    )
    
    father_sim_col = f"baptism_father_{spouse_role}_father_similarity"
    mother_sim_col = f"baptism_mother_{spouse_role}_mother_similarity"
    child_sim_col = f"child_{spouse_role}_similarity"
    
    candidates["parent_support_count"] = candidates.apply(
        lambda row: count_parent_support(row, father_sim_col, mother_sim_col),
        axis=1
    )
    
    # Decision category
    candidates["cross_event_decision"] = np.select(
        [
            (
                candidates["age_at_marriage"].between(12, 45, inclusive="both")
                & candidates[child_sim_col].ge(0.90)
                & candidates["parent_support_count"].ge(2)
            ),
            (
                candidates["age_at_marriage"].between(10, 70, inclusive="both")
                & candidates[child_sim_col].ge(0.85)
                & candidates["parent_support_count"].ge(1)
            ),
            (
                candidates["age_at_marriage"].lt(10)
                | candidates["age_at_marriage"].gt(70)
            )
        ],
        [
            f"high_confidence_baptized_child_to_marriage_{spouse_role}",
            f"possible_baptized_child_to_marriage_{spouse_role}_review",
            "unlikely_due_to_age_gap"
        ],
        default="review_needed"
    )
    
    return candidates

Generate baptized-child-to-marriage candidates

In [24]:
baptism_to_marriage_husband = build_baptism_to_marriage_spouse_candidates(
    baptism_family,
    marriage_couples,
    spouse_role="husband"
)

baptism_to_marriage_wife = build_baptism_to_marriage_spouse_candidates(
    baptism_family,
    marriage_couples,
    spouse_role="wife"
)

print("Baptism → Marriage Husband candidates:", baptism_to_marriage_husband.shape)
print("Baptism → Marriage Wife candidates:", baptism_to_marriage_wife.shape)

display(baptism_to_marriage_husband.head())
display(baptism_to_marriage_wife.head())

Baptism → Marriage Husband candidates: (1171, 77)
Baptism → Marriage Wife candidates: (1177, 77)


,event_idno_baptism,event_date_baptism,event_year_baptism,event_place_clean_baptism,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,age_at_marriage,age_at_marriage_category,child_husband_similarity,child_husband_relation,baptism_father_husband_father_similarity,baptism_father_husband_father_relation,baptism_mother_husband_mother_similarity,baptism_mother_husband_mother_relation,parent_support_count,cross_event_decision
0,bautizo-1170,1807-12-08,1807.0,pampamarca,simon aime,mariano aime,damiana huauia,persona-4542,persona-4543,persona-4544,...,16.0,possible_early_marriage,0.900000,close_variant,0.916667,close_variant,0.857143,same_first_name,2,high_confidence_baptized_child_to_marriage_hus...
1,bautizo-1397,1810-04-14,1810.0,aucara,mariano sanchez,bisente sanchez,maria ccoyllo,persona-5436,persona-5437,persona-5438,...,26.0,strong_plausible_marriage_age,0.933333,close_variant,0.933333,close_variant,0.960000,close_variant,2,high_confidence_baptized_child_to_marriage_hus...
2,bautizo-1536,1812-06-13,1812.0,ishua,asencio asto,damian asto,maria suycca,persona-5984,persona-5985,persona-5986,...,26.0,strong_plausible_marriage_age,1.000000,exact_same,1.000000,exact_same,0.869565,same_first_name,2,high_confidence_baptized_child_to_marriage_hus...
3,bautizo-1627,1813-08-11,1813.0,pampamarca,tiburcio urbano,antonio urbano,josefa huaya,persona-6339,persona-6340,persona-6341,...,21.0,strong_plausible_marriage_age,0.933333,close_variant,0.928571,close_variant,0.846154,same_first_name,2,high_confidence_baptized_child_to_marriage_hus...
4,bautizo-1763,1815-01-02,1815.0,chacralla,narciso sanches,julian sanches,cayetana huaya,persona-6877,persona-6878,persona-6879,...,23.0,strong_plausible_marriage_age,1.000000,exact_same,1.000000,exact_same,0.866667,same_first_name,2,high_confidence_baptized_child_to_marriage_hus...


,event_idno_baptism,event_date_baptism,event_year_baptism,event_place_clean_baptism,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,age_at_marriage,age_at_marriage_category,child_wife_similarity,child_wife_relation,baptism_father_wife_father_similarity,baptism_father_wife_father_relation,baptism_mother_wife_mother_similarity,baptism_mother_wife_mother_relation,parent_support_count,cross_event_decision
0,bautizo-1020,1804-08-12,1804.0,ishua,maria peral,manuel peral,maria guaman,persona-3957,persona-3958,persona-3959,...,104.0,unlikely_too_old,0.855000,same_first_name,0.692308,same_first_name,0.727273,same_first_name,0,unlikely_due_to_age_gap
1,bautizo-127,1792-02-22,1792.0,unknown,nicolasa huamani,leon huamani,rufina ramos,persona-477,persona-478,persona-479,...,25.0,strong_plausible_marriage_age,0.937500,close_variant,0.916667,close_variant,1.000000,exact_same,2,high_confidence_baptized_child_to_marriage_wife
2,bautizo-1332,1809-11-06,1809.0,aucara,maria salome ayme,lorenso ayme,josefa lliuya,persona-5180,persona-5181,persona-5182,...,15.0,possible_early_marriage,0.855000,same_first_name,0.916667,close_variant,0.670588,same_first_name,1,possible_baptized_child_to_marriage_wife_review
3,bautizo-1684,1814-04-03,1814.0,ishua,francisca paula guachaca,silvestre guachaca,nicolasa achamisa,persona-6563,persona-6564,persona-6565,...,21.0,strong_plausible_marriage_age,0.809524,close_variant,0.944444,close_variant,0.838710,close_variant,2,review_needed
4,bautizo-1774,1815-02-23,1815.0,pampamarca,petrona cruz,francisco cruz,victoria quispe,persona-6920,persona-6921,persona-6922,...,6.0,unlikely_too_young,0.916667,close_variant,0.928571,close_variant,1.000000,exact_same,2,unlikely_due_to_age_gap


Marriage couple later appears as baptism parents

In [25]:
def build_marriage_couple_to_baptism_parents(marriage_df, baptism_df):
    """
    Finds cases where:
    marriage husband + wife later appear as baptism father + mother.
    
    This is strong couple-continuity evidence.
    """
    
    m = marriage_df.copy()
    b = baptism_df.copy()
    
    # Exact full-name pair
    exact_candidates = pd.DataFrame()
    
    if "spouse_pair_key" in m.columns and "parent_pair_key" in b.columns:
        m_exact = m[m["spouse_pair_key"].notna()].copy()
        b_exact = b[b["parent_pair_key"].notna()].copy()
        
        exact_candidates = b_exact.merge(
            m_exact,
            left_on="parent_pair_key",
            right_on="spouse_pair_key",
            how="inner",
            suffixes=("_baptism", "_marriage")
        )
        
        if not exact_candidates.empty:
            exact_candidates["blocking_strategy"] = "exact_parent_spouse_pair"
    
    # First-name pair
    b["parent_first_pair_key"] = b.apply(
        lambda x: make_key(x.get("father_first_name"), x.get("mother_first_name")),
        axis=1
    )
    
    m["spouse_first_pair_key"] = m.apply(
        lambda x: make_key(x.get("husband_first_name"), x.get("wife_first_name")),
        axis=1
    )
    
    b_first = b[b["parent_first_pair_key"].notna()].copy()
    m_first = m[m["spouse_first_pair_key"].notna()].copy()
    
    first_candidates = b_first.merge(
        m_first,
        left_on="parent_first_pair_key",
        right_on="spouse_first_pair_key",
        how="inner",
        suffixes=("_baptism", "_marriage")
    )
    
    if not first_candidates.empty:
        first_candidates["blocking_strategy"] = "first_name_parent_spouse_pair"
    
    candidates = pd.concat(
        [exact_candidates, first_candidates],
        ignore_index=True
    )
    
    if candidates.empty:
        return candidates
    
    # Remove duplicate baptism-marriage pairs
    candidates["cross_pair_key"] = (
        candidates["event_idno_baptism"].astype(str)
        + " | "
        + candidates["event_idno_marriage"].astype(str)
        + " | marriage_couple_to_baptism_parents"
    )
    
    candidates = candidates.drop_duplicates(subset=["cross_pair_key"]).copy()
    
    # Gap: baptism after marriage
    candidates["baptism_after_marriage_gap"] = (
        pd.to_numeric(candidates["event_year_baptism"], errors="coerce")
        - pd.to_numeric(candidates["event_year_marriage"], errors="coerce")
    )
    
    candidates["baptism_after_marriage_category"] = candidates[
        "baptism_after_marriage_gap"
    ].apply(marriage_to_baptism_gap_category)
    
    # Compare husband to baptism father
    candidates = add_name_comparison(
        candidates,
        "husband_name",
        "father_name",
        "husband_father"
    )
    
    # Compare wife to baptism mother
    candidates = add_name_comparison(
        candidates,
        "wife_name",
        "mother_name",
        "wife_mother"
    )
    
    candidates["couple_similarity_avg"] = candidates[
        ["husband_father_similarity", "wife_mother_similarity"]
    ].mean(axis=1)
    
    candidates["cross_event_decision"] = np.select(
        [
            (
                candidates["baptism_after_marriage_gap"].between(0, 25, inclusive="both")
                & candidates["husband_father_similarity"].ge(0.90)
                & candidates["wife_mother_similarity"].ge(0.90)
            ),
            (
                candidates["baptism_after_marriage_gap"].between(-5, 40, inclusive="both")
                & candidates["husband_father_similarity"].ge(0.80)
                & candidates["wife_mother_similarity"].ge(0.80)
            ),
            (
                candidates["baptism_after_marriage_gap"].lt(-5)
                | candidates["baptism_after_marriage_gap"].gt(40)
            )
        ],
        [
            "high_confidence_marriage_couple_to_baptism_parents",
            "possible_marriage_couple_to_baptism_parents_review",
            "weak_due_to_unusual_year_gap"
        ],
        default="review_needed"
    )
    
    return candidates

Generate marriage-couple-to-baptism-parent candidates

In [26]:
marriage_couple_to_baptism_parents = build_marriage_couple_to_baptism_parents(
    marriage_couples,
    baptism_family
)

print(
    "Marriage couple → Baptism parents candidates:",
    marriage_couple_to_baptism_parents.shape
)

display(marriage_couple_to_baptism_parents.head())

Marriage couple → Baptism parents candidates: (2614, 73)


,event_idno_baptism,event_date_baptism,event_year_baptism,event_place_clean_baptism,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,spouse_first_pair_key,cross_pair_key,baptism_after_marriage_gap,baptism_after_marriage_category,husband_father_similarity,husband_father_relation,wife_mother_similarity,wife_mother_relation,couple_similarity_avg,cross_event_decision
0,bautizo-1941,1826-02-26,1826.0,aucara,paula guamani guamani,mariano guamani,gregoria condori,persona-7571,persona-7572,persona-7573,...,NaN,bautizo-1941 | matrimonio-140 | marriage_coupl...,3.0,strong_childbearing_window,1.0,exact_same,1.0,exact_same,1.0,high_confidence_marriage_couple_to_baptism_par...
1,bautizo-1945,1826-02-08,1826.0,aucara,juana ramos ramos,faustino ramos,bibiana quispe,persona-7587,persona-7588,persona-7589,...,NaN,bautizo-1945 | matrimonio-47 | marriage_couple...,8.0,strong_childbearing_window,1.0,exact_same,1.0,exact_same,1.0,high_confidence_marriage_couple_to_baptism_par...
2,bautizo-1950,1826-04-06,1826.0,aucara,josefa ramos ramos,tiburcio ramos,juana guamani,persona-7606,persona-7607,persona-7608,...,NaN,bautizo-1950 | matrimonio-69 | marriage_couple...,7.0,strong_childbearing_window,1.0,exact_same,1.0,exact_same,1.0,high_confidence_marriage_couple_to_baptism_par...
3,bautizo-1963,1826-06-19,1826.0,ishua santa iglesia,jose jerbacio santiago santiago,fermin santiago,angela urbano,persona-7656,persona-7657,persona-7658,...,NaN,bautizo-1963 | matrimonio-101 | marriage_coupl...,6.0,strong_childbearing_window,1.0,exact_same,1.0,exact_same,1.0,high_confidence_marriage_couple_to_baptism_par...
4,bautizo-1981,1827-12-22,1827.0,pampamarca santa iglesia,francisco nabica torres torres,ylario torres,juliana huamani,persona-7727,persona-7728,persona-7729,...,NaN,bautizo-1981 | matrimonio-95 | marriage_couple...,7.0,strong_childbearing_window,1.0,exact_same,1.0,exact_same,1.0,high_confidence_marriage_couple_to_baptism_par...


Burial review evidence

In [19]:
# Same deceased + same parents
same_deceased_parents = self_merge_pairs(
    burial_family,
    key_col="deceased_parent_key",
    review_name="SameDeceasedParents",
    max_group_size=MAX_GROUP_SIZE
)

if not same_deceased_parents.empty:
    same_deceased_parents = add_year_features(same_deceased_parents)
    same_deceased_parents = same_deceased_parents[same_deceased_parents["year_gap"] <= 5].copy()
    same_deceased_parents["burial_decision"] = "possible_duplicate_burial_same_deceased_same_parents"


# Same deceased + same spouse
same_deceased_spouse = self_merge_pairs(
    burial_family,
    key_col="deceased_spouse_key",
    review_name="SameDeceasedSpouse",
    max_group_size=MAX_GROUP_SIZE
)

if not same_deceased_spouse.empty:
    same_deceased_spouse = add_year_features(same_deceased_spouse)
    same_deceased_spouse = same_deceased_spouse[same_deceased_spouse["year_gap"] <= 5].copy()
    same_deceased_spouse["burial_decision"] = "possible_duplicate_burial_same_deceased_same_spouse"


# Different deceased + same parents
same_burial_parents = self_merge_pairs(
    burial_family,
    key_col="burial_parent_pair_key",
    review_name="SameBurialParents",
    max_group_size=MAX_GROUP_SIZE
)

if not same_burial_parents.empty:
    same_burial_parents = add_year_features(same_burial_parents)
    same_burial_parents = same_burial_parents[
        same_burial_parents["deceased_name"].notna() &
        same_burial_parents["deceased_name_r"].notna() &
        (same_burial_parents["deceased_name"] != same_burial_parents["deceased_name_r"]) &
        (same_burial_parents["year_gap"] <= 40)
    ].copy()
    same_burial_parents = add_name_comparison(
        same_burial_parents, "deceased_name", "deceased_name_r", "deceased"
    )
    same_burial_parents["burial_decision"] = "different_deceased_same_parents_possible_siblings"


print("Burial review evidence created.")

SameDeceasedParents: 43 review pairs created using deceased_parent_key
SameDeceasedSpouse: 36 review pairs created using deceased_spouse_key
SameBurialParents: 190 review pairs created using burial_parent_pair_key
Burial review evidence created.


Cross-event evidence: Baptism to Burial

In [20]:
# Create blocking keys for baptism-burial candidate links
baptism_cross = baptism_family.copy()
burial_cross = burial_family.copy()

baptism_cross["child_parent_first_key"] = baptism_cross.apply(
    lambda x: make_key(
        x.get("child_first_name"),
        x.get("father_first_name"),
        x.get("mother_first_name")
    ),
    axis=1
)

burial_cross["deceased_parent_first_key"] = burial_cross.apply(
    lambda x: make_key(
        x.get("deceased_first_name"),
        x.get("father_first_name"),
        x.get("mother_first_name")
    ),
    axis=1
)

baptism_burial_candidates = baptism_cross.merge(
    burial_cross,
    left_on="child_parent_first_key",
    right_on="deceased_parent_first_key",
    how="inner",
    suffixes=("_baptism", "_burial")
)

if not baptism_burial_candidates.empty:
    baptism_burial_candidates["life_gap"] = (
        pd.to_numeric(baptism_burial_candidates["event_year_burial"], errors="coerce") -
        pd.to_numeric(baptism_burial_candidates["event_year_baptism"], errors="coerce")
    )
    
    baptism_burial_candidates["burial_after_baptism"] = baptism_burial_candidates["life_gap"] >= 0
    
    baptism_burial_candidates = add_name_comparison(
        baptism_burial_candidates, "child_name", "deceased_name", "child_deceased"
    )
    baptism_burial_candidates = add_name_comparison(
        baptism_burial_candidates, "father_name_baptism", "father_name_burial", "father"
    )
    baptism_burial_candidates = add_name_comparison(
        baptism_burial_candidates, "mother_name_baptism", "mother_name_burial", "mother"
    )
    
    baptism_burial_candidates = baptism_burial_candidates[
        (baptism_burial_candidates["burial_after_baptism"] == True) &
        (baptism_burial_candidates["life_gap"] <= 120) &
        (baptism_burial_candidates["child_deceased_similarity"] >= 0.85) &
        (baptism_burial_candidates["father_similarity"] >= 0.80) &
        (baptism_burial_candidates["mother_similarity"] >= 0.80)
    ].copy()
    
    baptism_burial_candidates["cross_event_decision"] = "possible_baptism_to_burial_identity_link"

print("Baptism-Burial candidates:", baptism_burial_candidates.shape)
display(baptism_burial_candidates.head())

Baptism-Burial candidates: (54, 77)


,event_idno_baptism,event_date_baptism,event_year_baptism,event_place_clean_baptism,child_name,father_name_baptism,mother_name_baptism,child_persona_id,father_persona_id_baptism,mother_persona_id_baptism,...,deceased_parent_first_key,life_gap,burial_after_baptism,child_deceased_similarity,child_deceased_relation,father_similarity,father_relation,mother_similarity,mother_relation,cross_event_decision
92400,bautizo-2198,1825-03-27,1825.0,aucara,maria nolasco nolasco,antonio nolasco,carmen huamani,persona-8576,persona-8577,persona-8578,...,maria | antonio | carmen,90.0,True,0.855000,same_first_name,0.866667,same_first_name,1.000000,exact_same,possible_baptism_to_burial_identity_link
123481,bautizo-2734,1897-02-15,1897.0,aucara,benigna fuentez,saturnino fuentez,nicolasa zarabia,persona-10675,persona-10676,persona-10677,...,benigna | saturnino | nicolasa,14.0,True,0.965517,close_variant,0.941176,close_variant,0.875000,same_first_name,possible_baptism_to_burial_identity_link
132722,bautizo-3118,1889-01-05,1889.0,aucara,justina huarancca huarancca,luis huarancca,francisca ochoa,persona-12199,persona-12200,persona-12201,...,justina | luis | francisca,0.0,True,0.900000,close_variant,1.000000,exact_same,1.000000,exact_same,possible_baptism_to_burial_identity_link
132724,bautizo-3165,1889-05-10,1889.0,aucara,monica champa champa,luciano champa,gabriela ramos,persona-12387,persona-12388,persona-12389,...,monica | luciano | gabriela,30.0,True,0.900000,close_variant,1.000000,exact_same,1.000000,exact_same,possible_baptism_to_burial_identity_link
133565,bautizo-3217,1889-11-10,1889.0,aucara,toribia vellido vellido,mariano vellido,monica aivar,persona-12594,persona-12595,persona-12596,...,toribia | mariano | monica,5.0,True,0.855000,same_first_name,0.933333,close_variant,0.916667,close_variant,possible_baptism_to_burial_identity_link


Cross-event evidence: Marriage to Burial

In [21]:
marriage_cross = marriage_couples.copy()
burial_cross = burial_family.copy()

# Case A:
# Burial deceased is wife, burial husband matches marriage husband.
marriage_cross["husband_wife_first_key"] = marriage_cross.apply(
    lambda x: make_key(x.get("husband_first_name"), x.get("wife_first_name")),
    axis=1
)

burial_cross["husband_deceased_first_key"] = burial_cross.apply(
    lambda x: make_key(x.get("husband_first_name"), x.get("deceased_first_name")),
    axis=1
)

wife_burial_candidates = marriage_cross.merge(
    burial_cross,
    left_on="husband_wife_first_key",
    right_on="husband_deceased_first_key",
    how="inner",
    suffixes=("_marriage", "_burial")
)

if not wife_burial_candidates.empty:
    wife_burial_candidates["years_after_marriage"] = (
        pd.to_numeric(wife_burial_candidates["event_year_burial"], errors="coerce") -
        pd.to_numeric(wife_burial_candidates["event_year_marriage"], errors="coerce")
    )
    
    wife_burial_candidates = add_name_comparison(
        wife_burial_candidates, "husband_name_marriage", "husband_name_burial", "husband"
    )
    wife_burial_candidates = add_name_comparison(
        wife_burial_candidates, "wife_name", "deceased_name", "wife_deceased"
    )
    
    wife_burial_candidates = wife_burial_candidates[
        (wife_burial_candidates["years_after_marriage"] >= 0) &
        (wife_burial_candidates["years_after_marriage"] <= 80) &
        (wife_burial_candidates["husband_similarity"] >= 0.80) &
        (wife_burial_candidates["wife_deceased_similarity"] >= 0.85)
    ].copy()
    
    wife_burial_candidates["cross_event_decision"] = "possible_married_wife_to_burial_deceased_link"


# Case B:
# Burial deceased is husband, burial wife matches marriage wife.
burial_cross["deceased_wife_first_key"] = burial_cross.apply(
    lambda x: make_key(x.get("deceased_first_name"), x.get("wife_first_name")),
    axis=1
)

husband_burial_candidates = marriage_cross.merge(
    burial_cross,
    left_on="husband_wife_first_key",
    right_on="deceased_wife_first_key",
    how="inner",
    suffixes=("_marriage", "_burial")
)

if not husband_burial_candidates.empty:
    husband_burial_candidates["years_after_marriage"] = (
        pd.to_numeric(husband_burial_candidates["event_year_burial"], errors="coerce") -
        pd.to_numeric(husband_burial_candidates["event_year_marriage"], errors="coerce")
    )
    
    husband_burial_candidates = add_name_comparison(
        husband_burial_candidates, "husband_name_marriage", "deceased_name", "husband_deceased"
    )
    husband_burial_candidates = add_name_comparison(
        husband_burial_candidates, "wife_name_marriage", "wife_name_burial", "wife"
    )
    
    husband_burial_candidates = husband_burial_candidates[
        (husband_burial_candidates["years_after_marriage"] >= 0) &
        (husband_burial_candidates["years_after_marriage"] <= 80) &
        (husband_burial_candidates["husband_deceased_similarity"] >= 0.85) &
        (husband_burial_candidates["wife_similarity"] >= 0.80)
    ].copy()
    
    husband_burial_candidates["cross_event_decision"] = "possible_married_husband_to_burial_deceased_link"


print("Wife burial candidates:", wife_burial_candidates.shape)
print("Husband burial candidates:", husband_burial_candidates.shape)

display(wife_burial_candidates.head())
display(husband_burial_candidates.head())

Wife burial candidates: (0, 86)
Husband burial candidates: (69, 86)


,event_idno_marriage,event_date_marriage,event_year_marriage,event_place_clean_marriage,husband_name_marriage,husband_father_name,husband_mother_name,wife_name_marriage,wife_father_name,wife_mother_name,...,spouse_first_name,spouse_last_name,husband_deceased_first_key,years_after_marriage,husband_similarity,husband_relation,wife_name,wife_deceased_similarity,wife_deceased_relation,cross_event_decision


,event_idno_marriage,event_date_marriage,event_year_marriage,event_place_clean_marriage,husband_name_marriage,husband_father_name,husband_mother_name,wife_name_marriage,wife_father_name,wife_mother_name,...,spouse_first_name,spouse_last_name,husband_deceased_first_key,deceased_wife_first_key,years_after_marriage,husband_deceased_similarity,husband_deceased_relation,wife_similarity,wife_relation,cross_event_decision
4,matrimonio-1019,1863-06-02,1863.0,aucara,bruno sanches,NaN,NaN,manuela quispe,antonio quispe,norberta ayme,...,manuela,quizpe,NaN,bruno | manuela,8.0,1.000000,exact_same,0.928571,close_variant,possible_married_husband_to_burial_deceased_link
5,matrimonio-1019,1863-06-02,1863.0,aucara,bruno sanches,NaN,NaN,manuela quispe,antonio quispe,norberta ayme,...,manuela,quizpe,NaN,bruno | manuela,8.0,1.000000,exact_same,0.928571,close_variant,possible_married_husband_to_burial_deceased_link
11,matrimonio-1053,1868-08-07,1868.0,aucara,mariano vega,NaN,NaN,cipriana quizpe,NaN,NaN,...,cipriana,quispe,NaN,mariano | cipriana,25.0,1.000000,exact_same,0.933333,close_variant,possible_married_husband_to_burial_deceased_link
12,matrimonio-1060,1869-02-24,1869.0,aucara,cristoval huamani,NaN,NaN,francisca huallpatuero,NaN,NaN,...,francisca,huallpatuero,NaN,cristoval | francisca,2.0,1.000000,exact_same,1.000000,exact_same,possible_married_husband_to_burial_deceased_link
13,matrimonio-1065,1869-06-28,1869.0,aucara,santos ayquipa,NaN,NaN,severina quizpe,tomas unknown,maria espinoza,...,severina,quispe,NaN,santos | severina,27.0,0.896552,close_variant,0.933333,close_variant,possible_married_husband_to_burial_deceased_link


Create evidence summaries

In [28]:
def evidence_count(name, df):
    """
    Creates one summary row for each evidence dataframe.
    Handles normal self-merge review tables and cross-event tables.
    """

    if df is None:
        return {
            "evidence_table": name,
            "rows": 0,
            "unique_pairs": 0
        }

    if df.empty:
        return {
            "evidence_table": name,
            "rows": 0,
            "unique_pairs": 0
        }

    if "pair_key" in df.columns:
        unique_pairs = df["pair_key"].nunique()

    elif "cross_pair_key" in df.columns:
        unique_pairs = df["cross_pair_key"].nunique()

    else:
        unique_pairs = np.nan

    return {
        "evidence_table": name,
        "rows": len(df),
        "unique_pairs": unique_pairs
    }


evidence_summary = pd.DataFrame([
    # =========================
    # Baptism evidence
    # =========================
    evidence_count("Child_Duplicate_Candidates", child_duplicates),
    evidence_count("SameParentCouple", same_parent_couple),
    evidence_count("Father_SimilarMother", father_similar_mother),
    evidence_count("Mother_SimilarFather", mother_similar_father),
    evidence_count("Father_DifferentMother", father_different_mother),
    evidence_count("Mother_DifferentFather", mother_different_father),

    # =========================
    # Marriage evidence
    # =========================
    evidence_count("SameMarriageCouple", same_marriage_couple),
    evidence_count("Husband_SimilarWife", husband_similar_wife),
    evidence_count("Wife_SimilarHusband", wife_similar_husband),
    evidence_count("Husband_DifferentWife", husband_different_wife),
    evidence_count("Wife_DifferentHusband", wife_different_husband),

    # =========================
    # Burial evidence
    # =========================
    evidence_count("SameDeceasedParents", same_deceased_parents),
    evidence_count("SameDeceasedSpouse", same_deceased_spouse),
    evidence_count("SameBurialParents", same_burial_parents),

    # =========================
    # Baptism ↔ Marriage cross evidence
    # =========================
    evidence_count("Baptism_to_Marriage_Husband", baptism_to_marriage_husband),
    evidence_count("Baptism_to_Marriage_Wife", baptism_to_marriage_wife),
    evidence_count("MarriageCouple_to_BaptismParents", marriage_couple_to_baptism_parents),

    # =========================
    # Baptism ↔ Burial cross evidence
    # =========================
    evidence_count("Baptism_Burial_Candidates", baptism_burial_candidates),

    # =========================
    # Marriage ↔ Burial cross evidence
    # =========================
    evidence_count("Wife_Burial_Candidates", wife_burial_candidates),
    evidence_count("Husband_Burial_Candidates", husband_burial_candidates),
])

display(evidence_summary)

,evidence_table,rows,unique_pairs
0,Child_Duplicate_Candidates,21,21.0
1,SameParentCouple,1258,1258.0
2,Father_SimilarMother,1028,1028.0
3,Mother_SimilarFather,744,744.0
4,Father_DifferentMother,6339,6339.0
5,Mother_DifferentFather,3614,3614.0
6,SameMarriageCouple,12,12.0
7,Husband_SimilarWife,4,4.0
8,Wife_SimilarHusband,2,2.0
9,Husband_DifferentWife,221,221.0


Export review evidence to Excel

In [29]:
def safe_sheet_name(name):
    return name[:31]


def clean_for_excel(df):
    """
    Keeps Excel export safe and avoids huge workbook crashes.
    """
    if df is None or df.empty:
        return pd.DataFrame({"message": ["No records generated for this evidence category."]})
    
    return df.head(MAX_EXPORT_ROWS).copy()


def write_review_workbook(path, sheets):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            clean_df = clean_for_excel(df)
            clean_df.to_excel(writer, sheet_name=safe_sheet_name(sheet_name), index=False)


baptism_output = OUTPUT_DIR / "baptism_review_evidence_v2.xlsx"
marriage_output = OUTPUT_DIR / "marriage_review_evidence_v2.xlsx"
burial_output = OUTPUT_DIR / "burial_review_evidence_v2.xlsx"
cross_event_output = OUTPUT_DIR / "cross_event_review_evidence_v2.xlsx"
summary_output = OUTPUT_DIR / "family_review_evidence_summary.xlsx"

write_review_workbook(
    baptism_output,
    {
        "Summary": evidence_summary[evidence_summary["evidence_table"].str.contains(
            "Child|Parent|Father_|Mother_", regex=True
        )],
        "Child_Duplicates": child_duplicates,
        "SameParentCouple": same_parent_couple,
        "Father_SimilarMother": father_similar_mother,
        "Mother_SimilarFather": mother_similar_father,
        "Father_DifferentMother": father_different_mother,
        "Mother_DifferentFather": mother_different_father,
    }
)

write_review_workbook(
    marriage_output,
    {
        "Summary": evidence_summary[evidence_summary["evidence_table"].str.contains(
            "Marriage|Husband|Wife", regex=True
        )],
        "SameMarriageCouple": same_marriage_couple,
        "Husband_SimilarWife": husband_similar_wife,
        "Wife_SimilarHusband": wife_similar_husband,
        "Husband_DifferentWife": husband_different_wife,
        "Wife_DifferentHusband": wife_different_husband,
    }
)

write_review_workbook(
    burial_output,
    {
        "Summary": evidence_summary[evidence_summary["evidence_table"].str.contains(
            "Deceased|Burial", regex=True
        )],
        "SameDeceasedParents": same_deceased_parents,
        "SameDeceasedSpouse": same_deceased_spouse,
        "SameBurialParents": same_burial_parents,
    }
)

write_review_workbook(
    cross_event_output,
    {
        "Summary": evidence_summary[evidence_summary["evidence_table"].str.contains(
            "Candidates", regex=True
        )],
        "Baptism_Burial": baptism_burial_candidates,
        "Wife_Burial": wife_burial_candidates,
        "Husband_Burial": husband_burial_candidates,
    }
)

with pd.ExcelWriter(summary_output, engine="openpyxl") as writer:
    evidence_summary.to_excel(writer, sheet_name="Evidence_Counts", index=False)
    
    for name, large_df in large_group_reports.items():
        if large_df is not None and not large_df.empty:
            large_df.head(MAX_EXPORT_ROWS).to_excel(
                writer,
                sheet_name=safe_sheet_name(f"{name}_LargeGroups"),
                index=False
            )

print("Saved review evidence files:")
print(baptism_output)
print(marriage_output)
print(burial_output)
print(cross_event_output)
print(summary_output)

Saved review evidence files:
outputs\family_review_evidence\baptism_review_evidence_v2.xlsx
outputs\family_review_evidence\marriage_review_evidence_v2.xlsx
outputs\family_review_evidence\burial_review_evidence_v2.xlsx
outputs\family_review_evidence\cross_event_review_evidence_v2.xlsx
outputs\family_review_evidence\family_review_evidence_summary.xlsx


In [30]:
print("Final evidence summary:")
display(evidence_summary)

print("\nOutput files created in:")
print(OUTPUT_DIR.resolve())

Final evidence summary:


,evidence_table,rows,unique_pairs
0,Child_Duplicate_Candidates,21,21.0
1,SameParentCouple,1258,1258.0
2,Father_SimilarMother,1028,1028.0
3,Mother_SimilarFather,744,744.0
4,Father_DifferentMother,6339,6339.0
5,Mother_DifferentFather,3614,3614.0
6,SameMarriageCouple,12,12.0
7,Husband_SimilarWife,4,4.0
8,Wife_SimilarHusband,2,2.0
9,Husband_DifferentWife,221,221.0



Output files created in:
C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\family_review_evidence
